In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Flatten

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator



In [ ]:
def preprocess_image(img):

    lab = cv2.cvtColor(img.astype('uint8'), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    enhanced_img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


    noisy_img = enhanced_img + np.random.normal(0, 0.05 * 255, enhanced_img.shape)
    noisy_img = np.clip(noisy_img, 0, 255)
    return noisy_img

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,shear_range = 0.2,zoom_range = 0.2,horizontal_flip = True)
test_datagen = ImageDataGenerator(rescale = 1./255)

In [ ]:
x_train = train_datagen.flow_from_directory(directory="/content/drive/MyDrive/SI-GuidedProject-78335-1656736746-main/data/train",target_size = (64,64),batch_size = 32,class_mode = "categorical")
x_test = test_datagen.flow_from_directory(directory="/content/drive/MyDrive/SI-GuidedProject-78335-1656736746-main/data/test",target_size = (64,64),batch_size = 32,class_mode = "categorical")

Found 15341 images belonging to 6 classes.
Found 6848 images belonging to 6 classes.


In [ ]:
model = Sequential()

In [ ]:
from tensorflow.keras.layers import BatchNormalization

In [ ]:
model.add(Conv2D(32, (3, 3), input_shape=(64, 64, 1), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.add(Flatten())

In [ ]:
from tensorflow.keras.layers import Reshape

model.add(Reshape((16, 784)))

In [ ]:
from tensorflow.keras.layers import LSTM, Dropout

model.add(LSTM(128, activation="tanh", return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128, activation="tanh"))

In [ ]:

model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))

In [ ]:
model.add(Dense(units=6, activation="softmax"))

In [ ]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)                    │ (None, 62, 62, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 31, 31, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 14, 14, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 12544)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape_1 (Reshape)                  │ (None, 16, 784)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (None, 16, 128)             │         467,456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 16, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_4 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 128)                 │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 634,752 (2.42 MB)

 Trainable params: 634,560 (2.42 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
steps_per_epoch = len(x_train) // x_train.batch_size
validation_steps = len(x_test) // x_test.batch_size

In [ ]:
model.fit(
    x_train,
    epochs=30,
    validation_data=x_test,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps
)

Epoch 1/30


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


15/15 ━━━━━━━━━━━━━━━━━━━━ 561s 29s/step - accuracy: 0.4118 - loss: 1.5186 - val_accuracy: 0.3958 - val_loss: 1.5540
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 328s 23s/step - accuracy: 0.4948 - loss: 1.4294 - val_accuracy: 0.3229 - val_loss: 1.6531
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 237s 17s/step - accuracy: 0.4601 - loss: 1.4790 - val_accuracy: 0.2760 - val_loss: 1.7067
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 230s 16s/step - accuracy: 0.4436 - loss: 1.4715 - val_accuracy: 0.2708 - val_loss: 1.6587
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 271s 18s/step - accuracy: 0.4573 - loss: 1.4227 - val_accuracy: 0.3385 - val_loss: 1.5969
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 247s 17s/step - accuracy: 0.4743 - loss: 1.4207 - val_accuracy: 0.3073 - val_loss: 1.6218
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 246s 16s/step - accuracy: 0.5220 - loss: 1.2759 - val_accuracy: 0.4583 - val_loss: 1.5306
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 250s 17s/step - accuracy: 0.6635 - loss: 1.0296 - val_accuracy: 0.4583 - val_

In [ ]:
model.save('Hybrid_ECG_Model.h5')

In [ ]:
from google.colab import files

files.download('Hybrid_ECG_Model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

In [ ]:
model = load_model('Hybrid_ECG_Model.h5')
img = image.load_img("/content/drive/MyDrive/SI-GuidedProject-78335-1656736746-main/training/Unknown_image.png", target_size=(64, 64))

In [ ]:
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)

In [ ]:
pred = model.predict(x)
y_pred = np.argmax(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step


In [ ]:
index = ['left Bundle Branch block', 'Normal', 'Premature Atrial Contraction', 'Premature Ventricular Contraction', 'Right Bundle Branch Block', 'Ventricular Fibrillation']
result = str(index[y_pred])
print(result)

Normal
